# Notebook 06 — Pretrained ViT + Center-Aware Study Transformer

**Mục tiêu:** xây dựng lại mô hình ViT cho RSNA Knee Abnormality Detection theo ba thay đổi đã thống nhất:

1. Dùng **pretrained ViT-B/16 ImageNet-1K** làm image encoder; chỉ fine-tune hai Transformer blocks cuối ở giai đoạn đầu.
2. Giữ nguyên **224 × 224** của cache MRI, không resize về 160 × 160.
3. Sửa cách tính pseudo-label loss: sample weight không còn bị triệt tiêu khi chuẩn hóa loss. Kết hợp lấy mẫu có kiểm soát để pseudo-label không áp đảo gold label chỉ vì số lượng lớn.
4. Thử **soft center prior**: patch ở gần tâm ảnh được ưu tiên nhẹ trong attention pooling. Đây không phải hard mask/crop; các patch ngoài tâm vẫn có thể được attention học và chọn.

**Để so sánh công bằng:** notebook giữ nguyên `shared_split.csv`, labels và preprocessing cache từ Notebook 04 (CNN-LSTM). Validation vẫn là 12 gold studies cũ, vì vậy metric validation chỉ mang tính chẩn đoán và có độ bất định lớn.

> Quan trọng: notebook này là một thí nghiệm mới, không ghi đè checkpoint ViT cũ hoặc CNN-LSTM. Output được lưu ở `/kaggle/working/rsna_vit_pretrained_centerprior/`.

## 0. Kaggle setup và pretrained weights

Cần kết nối **Notebook Output/Dataset của Notebook 04 – CNN-LSTM training** để lấy shared artifacts.

- `cnn_lstm_model_artifacts.zip` (shared split, labels, preprocessing status)
- Training cache gồm đủ 6 file NPY. Cache có thể được tìm trong `/kaggle/input` **hoặc `/kaggle/working`** nếu đã được giữ lại nhờ File Persistence của notebook hiện tại.

Notebook ưu tiên cache đầy đủ trong `/kaggle/working` trước, sau đó mới tìm trong `/kaggle/input`. Không dùng `test_slices_uint8.npy` thay cho `train_slices_uint8.npy`.

Ngoài ra cần pretrained weights cho torchvision `vit_b_16` ImageNet-1K. Cell load weights sẽ:

1. Tìm file `vit_b_16-c867db91.pth` hoặc file có tên chứa `vit_b_16` trong `/kaggle/input`;
2. Tìm trong Torch Hub cache;
3. Nếu không thấy, thử tải bằng torchvision (chỉ thành công khi Internet được bật).

Nếu competition notebook không có Internet, hãy thêm pretrained checkpoint làm Kaggle Dataset/Model Input. Có thể điền đường dẫn cụ thể vào `PRETRAINED_WEIGHT_PATH` ở Cell 4. **Notebook sẽ dừng nếu không load được pretrained weights; tuyệt đối không âm thầm fallback về random initialization.**

Accelerator: GPU. Bộ nhớ khuyến nghị: T4 16 GB trở lên.

## 1. Imports và kiểm tra môi trường

In [ ]:
import os
import gc
import json
import time
import math
import random
import hashlib
import shutil
import zipfile
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision.models import vit_b_16, ViT_B_16_Weights

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

warnings.filterwarnings("ignore", category=UserWarning)

print("Python:", __import__("sys").version.split()[0])
print("PyTorch:", torch.__version__)
import torchvision
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU chưa bật. Kaggle Settings → Accelerator → GPU.")

device = torch.device("cuda")
for gpu_i in range(torch.cuda.device_count()):
    prop = torch.cuda.get_device_properties(gpu_i)
    print(f"GPU {gpu_i}: {prop.name} | {prop.total_memory / 1024**3:.1f} GiB")

## 2. Reproducibility, label order và experiment configuration

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

ID_COL = "StudyInstanceUID"
TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

EXPECTED_STUDIES = 4407
EXPECTED_GOLD = 58
EXPECTED_PSEUDO = 4349
MAX_SLICES = 36
IMAGE_SIZE = 224             # giữ nguyên resolution của preprocessing cache
PATCH_SIZE = 16              # ViT-B/16: 14 x 14 = 196 patches
EMBED_DIM = 256              # study-level representation
STUDY_DEPTH = 2
STUDY_HEADS = 4

# Pretrained encoder
PRETRAINED_WEIGHT_PATH = ""  # Có thể điền path cụ thể trong /kaggle/input nếu cần
TRY_TORCHVISION_DOWNLOAD = True
BACKBONE_NAME = "torchvision_vit_b_16_imagenet1k_v1"
FREEZE_EARLY_BLOCKS = 10      # freeze blocks 0..9, fine-tune blocks 10..11
SLICE_CHUNK_SIZE = 8          # giảm peak memory khi encode nhiều slices

# Weak, optional spatial prior at patch-attention pooling
USE_CENTER_PRIOR = True
CENTER_PRIOR_STRENGTH = 0.15  # nhỏ; không hard-mask vùng ngoại vi
CENTER_PRIOR_SIGMA = 0.55     # tọa độ chuẩn hóa [-1, 1]

# Corrected weak-label policy
GOLD_SAMPLE_WEIGHT = 1.0
PSEUDO_SAMPLE_WEIGHT = 0.05
PSEUDO_TO_GOLD_WEIGHTED_MASS_RATIO = 0.50
# Chọn pseudo subset mỗi epoch sao cho tổng pseudo weighted-label mass xấp xỉ
# 50% gold mass; tất cả gold-train studies được dùng mỗi epoch.

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_ROOT = (WORKING_ROOT / "rsna_vit_pretrained_pooling_v2")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Output:", OUTPUT_ROOT)
print("Targets:", len(TARGETS))
print("Image size:", IMAGE_SIZE, "| patch size:", PATCH_SIZE)
print("Center prior:", USE_CENTER_PRIOR, "strength:", CENTER_PRIOR_STRENGTH)

## 3. Helper functions để tìm Notebook Output và preprocessing cache

In [ ]:
SKIP_DIR_NAMES = {
    "train_series", "test_series", "train_images", "test_images",
    ".git", "__pycache__"
}

CACHE_REQUIRED_FILES = {
    "train_slices_uint8.npy", "slice_valid.npy", "plane_ids.npy",
    "fluid_flags.npy", "fat_flags.npy", "slice_positions.npy",
}
REQUIRED_ARTIFACTS = [
    "shared_split.csv",
    "study_labels_training.csv",
    "preprocess_status.csv",
]


def _unique_paths(paths):
    unique = {}
    for path in paths:
        try:
            key = str(Path(path).resolve())
        except OSError:
            key = str(Path(path))
        unique[key] = Path(path)
    return list(unique.values())


def find_named_files(root: Path, filename: str, max_depth: int = 10):
    """Find a file by case-insensitive basename under one root."""
    matches = []
    root = Path(root)
    if not root.exists():
        return matches
    for current, dirs, files in os.walk(root):
        current_path = Path(current)
        try:
            depth = len(current_path.relative_to(root).parts)
        except ValueError:
            depth = max_depth
        dirs[:] = [d for d in dirs if d not in SKIP_DIR_NAMES]
        for name in files:
            if name.lower() == filename.lower():
                matches.append(current_path / name)
        if depth >= max_depth:
            dirs.clear()
    return _unique_paths(matches)


def find_cache_directories(root: Path, max_depth: int = 10):
    """Find directories containing all six TRAIN cache arrays together."""
    found = []
    root = Path(root)
    if not root.exists():
        return found
    for current, dirs, files in os.walk(root):
        current_path = Path(current)
        try:
            depth = len(current_path.relative_to(root).parts)
        except ValueError:
            depth = max_depth
        dirs[:] = [d for d in dirs if d not in SKIP_DIR_NAMES]
        if CACHE_REQUIRED_FILES.issubset(set(files)):
            found.append(current_path)
        if depth >= max_depth:
            dirs.clear()
    return _unique_paths(found)


def find_extracted_artifact_directories(root: Path, max_depth: int = 10):
    """Find extracted shared-artifact folders (the three required CSVs together)."""
    found = []
    root = Path(root)
    if not root.exists():
        return found
    for current, dirs, files in os.walk(root):
        current_path = Path(current)
        try:
            depth = len(current_path.relative_to(root).parts)
        except ValueError:
            depth = max_depth
        dirs[:] = [d for d in dirs if d not in SKIP_DIR_NAMES]
        if set(REQUIRED_ARTIFACTS).issubset(set(files)):
            found.append(current_path)
        if depth >= max_depth:
            dirs.clear()
    return _unique_paths(found)


def common_path_depth(path_a: Path, path_b: Path):
    a = Path(path_a).resolve().parts
    b = Path(path_b).resolve().parts
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n

print("Kaggle input exists:", INPUT_ROOT.exists())
for p in INPUT_ROOT.iterdir():
    print(" -", p)
print("Kaggle working exists:", WORKING_ROOT.exists())


## 4. Tìm CNN-LSTM shared artifacts và training cache

In [ ]:
# CNN-LSTM shared artifacts may be mounted as a ZIP or already extracted.
artifact_zip_input_candidates = find_named_files(
    INPUT_ROOT, "cnn_lstm_model_artifacts.zip"
)
artifact_zip_working_candidates = find_named_files(
    WORKING_ROOT, "cnn_lstm_model_artifacts.zip"
)
# Prefer the mounted Kaggle Input archive; use the persistent working copy only
# when no archive is mounted in /kaggle/input.
artifact_zip_candidates = (
    artifact_zip_input_candidates
    if artifact_zip_input_candidates
    else artifact_zip_working_candidates
)
extracted_artifact_input_candidates = find_extracted_artifact_directories(INPUT_ROOT)
extracted_artifact_working_candidates = find_extracted_artifact_directories(WORKING_ROOT)
extracted_artifact_candidates = (
    extracted_artifact_input_candidates
    if extracted_artifact_input_candidates
    else extracted_artifact_working_candidates
)

# IMPORTANT: File Persistence can leave the six training NPY files in /kaggle/working.
# Search that location as well as /kaggle/input. The six files must be in ONE directory.
working_cache_candidates = find_cache_directories(WORKING_ROOT)
input_cache_candidates = find_cache_directories(INPUT_ROOT)

print("CNN-LSTM artifact ZIP candidates in /kaggle/input:")
for p in artifact_zip_input_candidates:
    print(" -", p)
print("CNN-LSTM artifact ZIP candidates in /kaggle/working:")
for p in artifact_zip_working_candidates:
    print(" -", p)
print("\nExtracted CNN-LSTM artifact directories selected for discovery:")
for p in extracted_artifact_candidates:
    print(" -", p)
print("\nTraining cache candidates in /kaggle/working (File Persistence):")
for p in working_cache_candidates:
    print(" -", p)
print("\nTraining cache candidates in /kaggle/input:")
for p in input_cache_candidates:
    print(" -", p)

if not artifact_zip_candidates and not extracted_artifact_candidates:
    raise FileNotFoundError(
        "Không tìm thấy CNN-LSTM shared artifacts. Cần cnn_lstm_model_artifacts.zip "
        "hoặc thư mục đã giải nén chứa đủ shared_split.csv, "
        "study_labels_training.csv và preprocess_status.csv."
    )

# Prefer the persistent cache in /kaggle/working when present.
# If several cache folders exist in the same location, stop rather than silently
# choosing a potentially stale cache. Set CACHE_DIR_OVERRIDE to one listed path.
CACHE_DIR_OVERRIDE = ""  # Optional: paste an exact cache directory path here.
if CACHE_DIR_OVERRIDE:
    CACHE_DIR = Path(CACHE_DIR_OVERRIDE)
    if not CACHE_REQUIRED_FILES.issubset({p.name for p in CACHE_DIR.glob("*.npy")}):
        raise FileNotFoundError(
            f"CACHE_DIR_OVERRIDE does not contain all six required arrays: {CACHE_DIR}"
        )
elif len(working_cache_candidates) == 1:
    CACHE_DIR = working_cache_candidates[0]
elif len(working_cache_candidates) > 1:
    raise RuntimeError(
        "Tìm thấy nhiều training cache trong /kaggle/working. "
        "Hãy đặt CACHE_DIR_OVERRIDE bằng đúng thư mục cache muốn dùng. Candidates: "
        + "; ".join(str(p) for p in working_cache_candidates)
    )
elif len(input_cache_candidates) == 1:
    CACHE_DIR = input_cache_candidates[0]
elif len(input_cache_candidates) > 1:
    raise RuntimeError(
        "Tìm thấy nhiều training cache trong /kaggle/input. "
        "Hãy đặt CACHE_DIR_OVERRIDE bằng đúng thư mục cache muốn dùng. Candidates: "
        + "; ".join(str(p) for p in input_cache_candidates)
    )
else:
    raise FileNotFoundError(
        "Không tìm thấy training cache đủ 6 file NPY trong cả /kaggle/working "
        "lẫn /kaggle/input. Cần train_slices_uint8.npy cùng 5 metadata arrays "
        "từ cùng một preprocessing run; không dùng test_slices_uint8.npy."
    )

# Prefer a single ZIP if present; otherwise use a single extracted artifact directory.
if len(artifact_zip_candidates) == 1:
    ARTIFACT_ZIP = artifact_zip_candidates[0]
    ARTIFACT_SOURCE_DIR = None
elif len(artifact_zip_candidates) > 1:
    raise RuntimeError(
        "Tìm thấy nhiều cnn_lstm_model_artifacts.zip trong cùng nguồn được ưu tiên. "
        "Hãy giữ lại đúng một bản. Candidates: "
        + "; ".join(str(p) for p in artifact_zip_candidates)
    )
elif len(extracted_artifact_candidates) == 1:
    ARTIFACT_ZIP = None
    ARTIFACT_SOURCE_DIR = extracted_artifact_candidates[0]
elif len(extracted_artifact_candidates) > 1:
    raise RuntimeError(
        "Tìm thấy nhiều thư mục shared artifacts trong cùng nguồn được ưu tiên. "
        "Candidates: " + "; ".join(str(p) for p in extracted_artifact_candidates)
    )
else:
    raise FileNotFoundError("Không tìm thấy CNN-LSTM ZIP hoặc extracted shared artifacts.")

print("\nSelected artifact ZIP:", ARTIFACT_ZIP)
print("Selected extracted artifact dir:", ARTIFACT_SOURCE_DIR)
print("Selected TRAIN cache:", CACHE_DIR)
for name in sorted(CACHE_REQUIRED_FILES):
    p = CACHE_DIR / name
    print(f"  {name}: {p.stat().st_size / 1024**3:.2f} GiB")


## 5. Giải nén shared artifacts

In [ ]:
ARTIFACT_DIR = OUTPUT_ROOT / "cnn_lstm_shared_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if ARTIFACT_ZIP is not None:
    with zipfile.ZipFile(ARTIFACT_ZIP, "r") as zf:
        bad_member = zf.testzip()
        if bad_member is not None:
            raise RuntimeError(f"Corrupt member in CNN artifact ZIP: {bad_member}")
        zf.extractall(ARTIFACT_DIR)
elif ARTIFACT_SOURCE_DIR is not None:
    for name in REQUIRED_ARTIFACTS:
        source_file = ARTIFACT_SOURCE_DIR / name
        if not source_file.is_file():
            raise FileNotFoundError(f"Missing shared artifact: {source_file}")
        shutil.copy2(source_file, ARTIFACT_DIR / name)
else:
    raise RuntimeError("No CNN-LSTM artifact source was selected in the discovery cell.")

missing = [name for name in REQUIRED_ARTIFACTS if not (ARTIFACT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"CNN-LSTM shared artifacts thiếu: {missing}")

for name in REQUIRED_ARTIFACTS:
    p = ARTIFACT_DIR / name
    print(f"{name}: {p.stat().st_size / 1024:.1f} KiB")


## 6. Load shared split, labels và preprocessing status

In [ ]:
shared_split = pd.read_csv(ARTIFACT_DIR / "shared_split.csv").fillna("")
label_table = pd.read_csv(ARTIFACT_DIR / "study_labels_training.csv")
preprocess_status = pd.read_csv(ARTIFACT_DIR / "preprocess_status.csv").fillna("")

for frame in [shared_split, label_table, preprocess_status]:
    if ID_COL not in frame.columns:
        raise ValueError(f"Missing column: {ID_COL}")
    frame[ID_COL] = frame[ID_COL].astype(str)

print("shared_split:", shared_split.shape)
print("study_labels_training:", label_table.shape)
print("preprocess_status:", preprocess_status.shape)
display(shared_split["split"].value_counts().rename_axis("split").reset_index(name="studies"))
display(preprocess_status["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="studies"))

## 7. Dựng label matrix và mask matrix; kiểm tra UID

In [ ]:
if not label_table[ID_COL].is_unique:
    raise ValueError("Duplicate StudyInstanceUID in label table")
if not shared_split[ID_COL].is_unique:
    raise ValueError("Duplicate StudyInstanceUID in split table")
if len(label_table) != EXPECTED_STUDIES:
    raise ValueError(f"Expected {EXPECTED_STUDIES} labels, got {len(label_table)}")

label_table = label_table.set_index(ID_COL)
shared_split = shared_split.set_index(ID_COL)
preprocess_status = preprocess_status.set_index(ID_COL)

uid_list = label_table.index.astype(str).tolist()
uid_to_idx = {uid: i for i, uid in enumerate(uid_list)}
if set(uid_list) != set(shared_split.index.astype(str)):
    raise ValueError("UID mismatch between labels and shared split")
if set(uid_list) != set(preprocess_status.index.astype(str)):
    raise ValueError("UID mismatch between labels and preprocessing status")

label_values = label_table[TARGETS].apply(pd.to_numeric, errors="coerce")
mask_matrix = label_values.notna().to_numpy(dtype=np.float32)
y_matrix = label_values.fillna(0.0).to_numpy(dtype=np.float32)
if not np.isin(y_matrix, [0.0, 1.0]).all():
    raise ValueError("Labels must be binary after masking missing/uncertain values")

source_names = label_table["label_source"].astype(str).to_numpy()
if set(np.unique(source_names)) - {"gold", "pseudo"}:
    raise ValueError(f"Unexpected label_source values: {np.unique(source_names)}")

# Use explicit policy rather than trusting an old exported sample_weight column.
sample_weights = np.where(
    source_names == "gold", GOLD_SAMPLE_WEIGHT, PSEUDO_SAMPLE_WEIGHT
).astype(np.float32)

split_roles = shared_split.loc[uid_list, "split"].astype(str).to_numpy()
status_values = preprocess_status.loc[uid_list, "status"].astype(str).to_numpy()

n_gold = int((source_names == "gold").sum())
n_pseudo = int((source_names == "pseudo").sum())
print("Gold:", n_gold, "| pseudo:", n_pseudo)
print("Observed target cells:", int(mask_matrix.sum()))
if n_gold != EXPECTED_GOLD or n_pseudo != EXPECTED_PSEUDO:
    raise ValueError(f"Expected gold/pseudo {EXPECTED_GOLD}/{EXPECTED_PSEUDO}, got {n_gold}/{n_pseudo}")

## 8. Load shared 224 × 224 preprocessing cache (memory-map)

In [ ]:
volume_path = CACHE_DIR / "train_slices_uint8.npy"
valid_path = CACHE_DIR / "slice_valid.npy"
plane_path = CACHE_DIR / "plane_ids.npy"
fluid_path = CACHE_DIR / "fluid_flags.npy"
fat_path = CACHE_DIR / "fat_flags.npy"
position_path = CACHE_DIR / "slice_positions.npy"

volumes = np.load(volume_path, mmap_mode="r")
valid_mm = np.load(valid_path, mmap_mode="r")
plane_mm = np.load(plane_path, mmap_mode="r")
fluid_mm = np.load(fluid_path, mmap_mode="r")
fat_mm = np.load(fat_path, mmap_mode="r")
position_mm = np.load(position_path, mmap_mode="r")

print("Images:", volumes.shape, volumes.dtype)
print("Valid:", valid_mm.shape, valid_mm.dtype)
print("Plane:", plane_mm.shape, plane_mm.dtype)
print("Fluid/Fat/Position:", fluid_mm.shape, fat_mm.shape, position_mm.shape)

expected_shape = (EXPECTED_STUDIES, MAX_SLICES, IMAGE_SIZE, IMAGE_SIZE)
if volumes.shape != expected_shape:
    raise ValueError(f"Expected image cache {expected_shape}, got {volumes.shape}")
if volumes.dtype != np.uint8:
    raise TypeError(f"Expected uint8 image cache, got {volumes.dtype}")
for arr in [valid_mm, plane_mm, fluid_mm, fat_mm, position_mm]:
    if arr.shape != volumes.shape[:2]:
        raise ValueError("A metadata cache has mismatched dimensions")

cache_gib = sum(p.stat().st_size for p in CACHE_DIR.glob("*.npy")) / 1024**3
print(f"Cache total size: {cache_gib:.2f} GiB")

## 9. Giữ nguyên train/validation split của CNN-LSTM

In [ ]:
role_to_ids = {
    role: [uid for uid, value in zip(uid_list, split_roles) if value == role]
    for role in np.unique(split_roles)
}
cache_ok_ids = {
    uid for uid, status in zip(uid_list, status_values) if status == "ok"
}

gold_train_ids = set(role_to_ids.get("gold_train", []))
gold_val_ids = set(role_to_ids.get("gold_validation", []))
pseudo_train_ids = set(role_to_ids.get("pseudo_train", []))

if len(gold_train_ids) + len(gold_val_ids) != EXPECTED_GOLD:
    raise ValueError("Gold train/validation does not cover all 58 gold studies")
if len(gold_val_ids) != 12:
    raise ValueError(f"Expected the same 12-study validation split, got {len(gold_val_ids)}")
if gold_train_ids & gold_val_ids:
    raise ValueError("Gold train and validation overlap")

missing_val = sorted(gold_val_ids - cache_ok_ids)
if missing_val:
    raise RuntimeError(f"Missing validation cache for {len(missing_val)} gold studies")

gold_train_ids = sorted(gold_train_ids & cache_ok_ids)
pseudo_train_ids = sorted(pseudo_train_ids & cache_ok_ids)
val_study_ids = sorted(gold_val_ids)

# Ignore studies that have no observed labels.
gold_train_ids = [u for u in gold_train_ids if mask_matrix[uid_to_idx[u]].sum() > 0]
pseudo_train_ids = [u for u in pseudo_train_ids if mask_matrix[uid_to_idx[u]].sum() > 0]

assert set(gold_train_ids).isdisjoint(set(val_study_ids))
assert all(source_names[uid_to_idx[u]] == "gold" for u in gold_train_ids + val_study_ids)
assert all(source_names[uid_to_idx[u]] == "pseudo" for u in pseudo_train_ids)

print("Gold train:", len(gold_train_ids))
print("Gold validation:", len(val_study_ids))
print("Pseudo trainable/cached:", len(pseudo_train_ids))
print("Validation source: same shared gold-only split as CNN-LSTM")

## 10. Visual sanity check: raw cached MRI slices

In [ ]:
example_uids = gold_train_ids[:2] + val_study_ids[:1]
fig, axes = plt.subplots(len(example_uids), 3, figsize=(10, 3.2 * len(example_uids)))
if len(example_uids) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, uid in enumerate(example_uids):
    idx = uid_to_idx[uid]
    valid_idx = np.flatnonzero(np.asarray(valid_mm[idx]).astype(bool))
    chosen = valid_idx[np.linspace(0, len(valid_idx) - 1, 3).astype(int)] if len(valid_idx) else [0, 0, 0]
    for col, slice_idx in enumerate(chosen):
        axes[row, col].imshow(volumes[idx, slice_idx], cmap="gray", vmin=0, vmax=255)
        axes[row, col].set_title(f"{uid[:8]} | slice {slice_idx}")
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

## 11. Study-level Dataset

In [ ]:
class StudyMRIDataset(Dataset):
    def __init__(self, study_ids):
        self.study_ids = list(study_ids)
        self.indices = np.asarray([uid_to_idx[u] for u in self.study_ids], dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        row_idx = int(self.indices[item])
        return {
            "images": torch.from_numpy(np.array(volumes[row_idx], copy=True)),
            "valid": torch.from_numpy(np.array(valid_mm[row_idx], copy=True).astype(np.uint8)),
            "plane_ids": torch.from_numpy(np.array(plane_mm[row_idx], copy=True).astype(np.int64)),
            "fluid": torch.from_numpy(np.array(fluid_mm[row_idx], copy=True).astype(np.float32)),
            "fat": torch.from_numpy(np.array(fat_mm[row_idx], copy=True).astype(np.float32)),
            "positions": torch.from_numpy(np.array(position_mm[row_idx], copy=True).astype(np.float32)),
            "labels": torch.from_numpy(y_matrix[row_idx].astype(np.float32)),
            "label_mask": torch.from_numpy(mask_matrix[row_idx].astype(np.float32)),
            "sample_weight": torch.tensor(sample_weights[row_idx], dtype=torch.float32),
            "uid": self.study_ids[item],
        }

print("Dataset ready. Example:", gold_train_ids[0])

## 12. Load pretrained ViT-B/16 weights

Backbone này nhận input 224 × 224 và dùng patch 16 × 16, tương ứng 14 × 14 = 196 patch tokens. Ảnh grayscale sẽ được lặp thành 3 channels rồi ImageNet-normalize.

Nếu không có local weights và Kaggle Internet bị tắt, cell sẽ dừng với thông báo cụ thể. Không được tiếp tục bằng encoder khởi tạo ngẫu nhiên.

In [ ]:
def find_local_vit_b16_weights():
    manual = Path(PRETRAINED_WEIGHT_PATH) if PRETRAINED_WEIGHT_PATH else None
    if manual is not None and manual.is_file():
        return manual

    hub_candidate = Path(torch.hub.get_dir()) / "checkpoints" / "vit_b_16-c867db91.pth"
    if hub_candidate.is_file():
        return hub_candidate

    matches = []
    for current, dirs, files in os.walk(INPUT_ROOT):
        current_path = Path(current)
        depth = len(current_path.relative_to(INPUT_ROOT).parts)
        dirs[:] = [d for d in dirs if d not in SKIP_DIR_NAMES]
        for name in files:
            lower = name.lower()
            if lower.endswith((".pth", ".pt")) and (
                "vit_b_16" in lower or "vit-b-16" in lower or "c867db91" in lower
            ):
                matches.append(current_path / name)
        if depth >= 9:
            dirs.clear()
    return matches[0] if matches else None


def unwrap_state_dict(obj):
    state = obj
    if isinstance(state, dict):
        for key in ["state_dict", "model_state_dict", "model_state", "model", "net"]:
            if key in state and isinstance(state[key], dict):
                state = state[key]
                break
    if not isinstance(state, dict):
        raise TypeError("Pretrained checkpoint does not contain a state_dict")

    # Remove common wrappers only when they are present on all tensor keys.
    tensor_keys = [k for k, v in state.items() if torch.is_tensor(v)]
    if tensor_keys and all(k.startswith("module.") for k in tensor_keys):
        state = {k[len("module."):]: v for k, v in state.items()}
    tensor_keys = [k for k, v in state.items() if torch.is_tensor(v)]
    if tensor_keys and all(k.startswith("backbone.") for k in tensor_keys):
        state = {k[len("backbone."):]: v for k, v in state.items()}
    return {k: v for k, v in state.items() if torch.is_tensor(v)}


PRETRAINED_BACKBONE = vit_b_16(weights=None)
local_weight_path = find_local_vit_b16_weights()

if local_weight_path is not None:
    print("Loading local pretrained weights:", local_weight_path)
    try:
        state_obj = torch.load(local_weight_path, map_location="cpu", weights_only=False)
    except TypeError:
        state_obj = torch.load(local_weight_path, map_location="cpu")
    pretrained_state = unwrap_state_dict(state_obj)
    incompat = PRETRAINED_BACKBONE.load_state_dict(pretrained_state, strict=False)
    critical_missing = [
        k for k in incompat.missing_keys
        if not k.startswith("heads.head.")
    ]
    critical_unexpected = [
        k for k in incompat.unexpected_keys
        if not k.startswith(("head.", "heads."))
    ]
    if critical_missing or critical_unexpected:
        raise RuntimeError(
            "Local file không phải torchvision ViT-B/16 compatible checkpoint.\n"
            f"Critical missing: {critical_missing[:12]}\n"
            f"Unexpected: {critical_unexpected[:12]}"
        )
    PRETRAINED_SOURCE = str(local_weight_path)
else:
    if not TRY_TORCHVISION_DOWNLOAD:
        raise FileNotFoundError(
            "Không tìm thấy pretrained ViT-B/16. Thêm file torchvision vit_b_16 weights "
            "làm Kaggle Input hoặc đặt TRY_TORCHVISION_DOWNLOAD=True khi có Internet."
        )
    print("Không tìm thấy local weights. Thử tải torchvision ImageNet-1K weights...")
    try:
        PRETRAINED_BACKBONE = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        PRETRAINED_SOURCE = "torchvision:ViT_B_16_Weights.IMAGENET1K_V1"
    except Exception as exc:
        raise RuntimeError(
            "Không tải được pretrained ViT-B/16. Có thể Kaggle Internet đang tắt. "
            "Hãy thêm torchvision vit_b_16-c867db91.pth làm Kaggle Dataset/Model Input, "
            "sau đó điền path vào PRETRAINED_WEIGHT_PATH. Không chạy random initialization."
        ) from exc

PRETRAINED_BACKBONE.eval()
print("Pretrained source:", PRETRAINED_SOURCE)
print("Backbone parameters:", f"{sum(p.numel() for p in PRETRAINED_BACKBONE.parameters()) / 1e6:.2f}M")

## 13. Slice encoder: pretrained ViT + soft center-aware patch pooling

**Center prior được áp dụng ở đâu?** Sau khi ViT tạo ra 196 patch tokens, một attention pooling layer học điểm số cho từng patch. Ta cộng thêm một Gaussian bias rất nhỏ vào điểm số đó:

\[
 b(x,y)=\lambda\left[-\frac{(x-0.5)^2+(y-0.5)^2}{2\sigma^2}\right]
\]

Điểm càng gần tâm thì bias càng cao. Vì đây chỉ là bias cộng vào learned attention scores, model vẫn có thể chọn patch ngoại vi nếu nội dung ảnh quan trọng. Không xóa, crop hay che vùng ngoại vi.

`FREEZE_EARLY_BLOCKS=10`: 10 blocks đầu được giữ frozen; 2 blocks cuối và LayerNorm cuối được fine-tune. Các patch/CLS tokens vẫn lấy từ cùng pretrained ViT-B/16.

In [ ]:
class PretrainedSliceEncoder(nn.Module):
    def __init__(
        self,
        backbone,
        output_dim=EMBED_DIM,
        freeze_early_blocks=FREEZE_EARLY_BLOCKS,
        use_center_prior=USE_CENTER_PRIOR,
        center_prior_strength=CENTER_PRIOR_STRENGTH,
        center_prior_sigma=CENTER_PRIOR_SIGMA,
        dropout=0.10,
    ):
        super().__init__()
        self.backbone = backbone
        self.hidden_dim = 768
        self.freeze_early_blocks = int(freeze_early_blocks)
        self.use_center_prior = bool(use_center_prior)

        blocks = list(self.backbone.encoder.layers.children())
        if not 0 <= self.freeze_early_blocks < len(blocks):
            raise ValueError(f"freeze_early_blocks must be in [0, {len(blocks)-1}]")

        # Freeze the entire pretrained backbone first.
        for p in self.backbone.parameters():
            p.requires_grad = False
        # Fine-tune only the last K transformer blocks and final LayerNorm.
        for block in blocks[self.freeze_early_blocks:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.backbone.encoder.ln.parameters():
            p.requires_grad = True

        self.trainable_blocks = blocks[self.freeze_early_blocks:]
        self.patch_attention = nn.Sequential(
            nn.LayerNorm(self.hidden_dim),
            nn.Linear(self.hidden_dim, 1, bias=False),
        )
        self.feature_projection = nn.Sequential(
            nn.Linear(self.hidden_dim * 2, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        grid = IMAGE_SIZE // PATCH_SIZE
        coords = torch.linspace(-1.0, 1.0, grid)
        yy, xx = torch.meshgrid(coords, coords, indexing="ij")
        sigma = max(float(center_prior_sigma), 1e-4)
        log_gaussian = -0.5 * ((xx / sigma) ** 2 + (yy / sigma) ** 2)
        log_gaussian = log_gaussian - log_gaussian.max()
        strength = float(center_prior_strength) if use_center_prior else 0.0
        self.register_buffer("center_bias", (strength * log_gaussian.reshape(-1)).float())

    def train(self, mode=True):
        # Keep frozen pretrained blocks in eval mode; enable train mode only on unfrozen blocks.
        super().train(mode)
        self.backbone.eval()
        for block in self.trainable_blocks:
            block.train(mode)
        self.backbone.encoder.ln.train(mode)
        self.patch_attention.train(mode)
        self.feature_projection.train(mode)
        return self

    def forward(self, images_rgb):
        # images_rgb: [N, 3, 224, 224], ImageNet-normalized.
        x = self.backbone._process_input(images_rgb)  # [N, 196, 768]
        cls = self.backbone.class_token.expand(x.shape[0], -1, -1)
        x = torch.cat([cls, x], dim=1)                 # [N, 197, 768]
        x = x + self.backbone.encoder.pos_embedding
        x = self.backbone.encoder.dropout(x)

        for block_i, block in enumerate(self.backbone.encoder.layers.children()):
            if block_i < self.freeze_early_blocks:
                with torch.no_grad():
                    x = block(x)
            else:
                x = block(x)

        x = self.backbone.encoder.ln(x)
        cls_feature = x[:, 0]
        patch_tokens = x[:, 1:]

        patch_scores = self.patch_attention(patch_tokens).squeeze(-1)
        patch_scores = patch_scores + self.center_bias.unsqueeze(0)
        patch_weights = torch.softmax(patch_scores, dim=1)
        spatial_feature = torch.sum(patch_tokens * patch_weights.unsqueeze(-1), dim=1)

        return self.feature_projection(torch.cat([cls_feature, spatial_feature], dim=-1))

print("Pretrained slice encoder defined.")
print("Trainable ViT blocks:", FREEZE_EARLY_BLOCKS, "..", 11)

## 14. Study-level Transformer: kết hợp 36 slice embeddings

In [ ]:

class StudyFeatureAggregator(nn.Module):
    """
    Study aggregation using:
    - Gated metadata embeddings
    - Slice-level Transformer contextualization
    - Masked mean pooling
    - Learned attention pooling
    - Residual connection from original slice features

    No study CLS token.
    """

    def __init__(
        self,
        embed_dim=EMBED_DIM,
        max_slices=MAX_SLICES,
        study_depth=STUDY_DEPTH,
        num_heads=STUDY_HEADS,
        dropout=0.10,
    ):
        super().__init__()

        self.plane_embedding = nn.Embedding(
            4, embed_dim, padding_idx=3
        )

        self.metadata_projection = nn.Sequential(
            nn.Linear(3, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
        )

        self.slice_order_embedding = nn.Embedding(
            max_slices, embed_dim
        )

        # Start metadata contribution at a small magnitude.
        self.metadata_scale = nn.Parameter(
            torch.tensor(0.10)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.study_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=study_depth,
            enable_nested_tensor=False,
        )

        # Learned slice attention pooling.
        self.attention_score = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 1),
        )

        # Combine contextual mean and attention pooling.
        self.pool_projection = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Direct residual path from original slice features.
        self.skip_scale = nn.Parameter(
            torch.tensor(0.5)
        )

        self.output_norm = nn.LayerNorm(embed_dim)

    def forward(
        self,
        slice_features,
        valid,
        plane_ids,
        fluid,
        fat,
        positions,
    ):

        batch_size, num_slices, embed_dim = slice_features.shape

        valid = valid.bool()

        if not valid.any(dim=1).all():
            raise ValueError(
                "Every study must contain at least one valid slice."
            )

        plane_ids = plane_ids.clamp(0, 3)

        metadata = torch.stack([
            fluid.clamp(-1, 1),
            fat.clamp(-1, 1),
            positions.clamp(0, 1),
        ], dim=-1)

        order_ids = torch.arange(
            num_slices,
            device=slice_features.device,
        ).unsqueeze(0)

        # Gated metadata injection.
        metadata_features = (
            self.plane_embedding(plane_ids)
            + self.metadata_projection(metadata)
            + self.slice_order_embedding(order_ids)
        )

        tokens = (
            slice_features
            + self.metadata_scale * metadata_features
        )

        # Contextualize slices without a global CLS token.
        encoded = self.study_encoder(
            tokens,
            src_key_padding_mask=~valid,
        )

        mask = valid.unsqueeze(-1).to(encoded.dtype)
        denominator = mask.sum(dim=1).clamp_min(1.0)

        # 1. Masked mean pooling.
        mean_pool = (
            encoded * mask
        ).sum(dim=1) / denominator

        # 2. Learned attention pooling.
        scores = self.attention_score(
            encoded
        ).squeeze(-1)

        scores = scores.masked_fill(
            ~valid,
            torch.finfo(scores.dtype).min,
        )

        attention_weights = torch.softmax(
            scores,
            dim=1,
        )

        attention_pool = torch.sum(
            encoded * attention_weights.unsqueeze(-1),
            dim=1,
        )

        # 3. Combine both pooling strategies.
        pooled = self.pool_projection(
            torch.cat(
                [mean_pool, attention_pool],
                dim=-1,
            )
        )

        # 4. Preserve direct information from original slices.
        raw_mean = (
            slice_features * mask
        ).sum(dim=1) / denominator

        study_features = self.output_norm(
            pooled + self.skip_scale * raw_mean
        )

        return study_features


print("New Residual Attention Study Aggregator defined.")


## 15. Full pretrained ViT study classifier

In [ ]:
class PretrainedViTStudyModel(nn.Module):
    def __init__(self, backbone, num_targets=len(TARGETS)):
        super().__init__()
        self.slice_encoder = PretrainedSliceEncoder(backbone=backbone)
        self.aggregator = StudyFeatureAggregator()
        self.classifier = nn.Sequential(
            nn.Linear(EMBED_DIM, 128),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(128, num_targets),
        )
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        self.register_buffer("image_mean", mean)
        self.register_buffer("image_std", std)

    def forward(self, images, valid, plane_ids, fluid, fat, positions):
        # images: [B, 36, 224, 224] uint8. No spatial resizing is performed.
        batch_size, num_slices, height, width = images.shape
        if height != IMAGE_SIZE or width != IMAGE_SIZE:
            raise ValueError(f"Expected {IMAGE_SIZE}x{IMAGE_SIZE}; got {height}x{width}")

        valid_flat = valid.bool().reshape(-1)
        valid_indices = valid_flat.nonzero(as_tuple=False).squeeze(1)
        if valid_indices.numel() == 0:
            raise ValueError("Batch contains no valid MRI slices")

        flat = images.reshape(batch_size * num_slices, 1, height, width)
        selected = flat.index_select(0, valid_indices).float().div(255.0)
        selected = selected.repeat(1, 3, 1, 1)
        selected = (selected - self.image_mean) / self.image_std

        # Chunked encoding controls temporary GPU memory; patch resolution stays 224x224.
        encoded_chunks = []
        for start in range(0, selected.shape[0], SLICE_CHUNK_SIZE):
            chunk = selected[start:start + SLICE_CHUNK_SIZE]
            encoded_chunks.append(self.slice_encoder(chunk))
        valid_features = torch.cat(encoded_chunks, dim=0)

        flat_features = valid_features.new_zeros((batch_size * num_slices, EMBED_DIM))
        flat_features = flat_features.index_copy(0, valid_indices, valid_features)
        slice_features = flat_features.view(batch_size, num_slices, EMBED_DIM)

        study_features = self.aggregator(
            slice_features, valid.bool(), plane_ids, fluid, fat, positions
        )
        return self.classifier(study_features)

model = PretrainedViTStudyModel(backbone=PRETRAINED_BACKBONE).to(device)

n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {n_total / 1e6:.2f}M")
print(f"Trainable parameters: {n_trainable / 1e6:.2f}M")

## 16. Visualize soft center prior (không phải hard crop)

In [ ]:
center_bias = model.slice_encoder.center_bias.detach().float().cpu().numpy().reshape(
    IMAGE_SIZE // PATCH_SIZE, IMAGE_SIZE // PATCH_SIZE
)
plt.figure(figsize=(5, 4))
plt.imshow(center_bias, cmap="viridis")
plt.colorbar(label="Additive attention bias")
plt.title("Soft center prior on the 14×14 patch grid")
plt.xlabel("Patch column")
plt.ylabel("Patch row")
plt.tight_layout()
plt.show()

print("Bias range:", float(center_bias.min()), "to", float(center_bias.max()))
print("Outer patches remain available to learned attention; no patches are masked.")

## 17. Smoke test: forward pass và kiểm tra output shape

In [ ]:
smoke_batch = next(iter(DataLoader(
    StudyMRIDataset(val_study_ids[:1]), batch_size=1, shuffle=False, num_workers=0
)))
model.eval()
with torch.no_grad(), torch.cuda.amp.autocast(enabled=True):
    smoke_logits = model(
        smoke_batch["images"].to(device),
        smoke_batch["valid"].to(device),
        smoke_batch["plane_ids"].to(device),
        smoke_batch["fluid"].to(device),
        smoke_batch["fat"].to(device),
        smoke_batch["positions"].to(device),
    )
print("Logits shape:", tuple(smoke_logits.shape))
assert smoke_logits.shape == (1, len(TARGETS))
assert torch.isfinite(smoke_logits).all()
del smoke_batch, smoke_logits
gc.collect()
torch.cuda.empty_cache()

## 18. Training configuration

Hai learning rates khác nhau:

- Pretrained ViT cuối: LR nhỏ để tránh phá hủy feature đã học trên ImageNet.
- Study aggregator, patch pooling và classifier: LR cao hơn vì khởi tạo mới.

PATIENCE được tăng lên 4 để tránh dừng quá sớm chỉ sau hai epoch không cải thiện. Tuy vậy, validation có 12 studies nên không nên xem việc chạy nhiều epoch hơn là mục tiêu tự thân.

In [ ]:
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 2
NUM_WORKERS = 2
EPOCHS = 10
LR_BACKBONE = 1e-5
LR_HEAD = 1e-4
WEIGHT_DECAY = 0.03
EARLY_STOPPING_PATIENCE = 4
MIN_DELTA = 1e-4
USE_AMP = True
MAX_GRAD_NORM = 1.0
RESUME_TRAINING = True

print("Batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRAD_ACCUM_STEPS)
print("Effective study batch:", BATCH_SIZE * GRAD_ACCUM_STEPS)
print("Epochs:", EPOCHS, "| patience:", EARLY_STOPPING_PATIENCE)
print("Backbone/head LR:", LR_BACKBONE, LR_HEAD)

## 19. Pseudo-label sampler: hạn chế số lượng pseudo áp đảo gold

Trong data có 46 gold-train studies và hàng nghìn pseudo-train studies. Nếu đưa tất cả vào mỗi epoch, gold chỉ chiếm xấp xỉ 1% examples. Dù gán pseudo weight thấp, gold gradients vẫn xuất hiện rất thưa.

Thiết kế epoch mới:

- Dùng **toàn bộ gold-train studies một lần mỗi epoch**.
- Lấy một subset pseudo không lặp trong epoch.
- Số pseudo được tính từ số target cells đang được mask, để tổng `pseudo_weight × observed_pseudo_cells` xấp xỉ 50% gold observed cells.
- Pseudo subset được lấy lại theo seed mỗi epoch; sau nhiều epoch model dần nhìn thấy nhiều pseudo studies khác nhau.

Đây là cách kiểm soát cả **sample weight** lẫn **tần suất xuất hiện**, không chỉ nhân loss rồi chia lại cho tổng weight.

In [ ]:
# Build one training dataset: gold studies first, then pseudo studies.
train_pool_ids = list(gold_train_ids) + list(pseudo_train_ids)
train_pool_local_gold = np.arange(len(gold_train_ids), dtype=np.int64)
train_pool_local_pseudo = np.arange(
    len(gold_train_ids), len(train_pool_ids), dtype=np.int64
)
train_pool_global_idx = np.asarray([uid_to_idx[u] for u in train_pool_ids], dtype=np.int64)

# Estimate pseudo subset size from observed (non-masked) target cells.
gold_global_idx = np.asarray([uid_to_idx[u] for u in gold_train_ids], dtype=np.int64)
pseudo_global_idx = np.asarray([uid_to_idx[u] for u in pseudo_train_ids], dtype=np.int64)
gold_observed_cells = float(mask_matrix[gold_global_idx].sum())
pseudo_observed_per_study = mask_matrix[pseudo_global_idx].sum(axis=1)
avg_pseudo_observed_cells = float(pseudo_observed_per_study.mean())

if gold_observed_cells <= 0 or avg_pseudo_observed_cells <= 0:
    raise ValueError("No observed training label cells to sample")

pseudo_weighted_cell_target = PSEUDO_TO_GOLD_WEIGHTED_MASS_RATIO * gold_observed_cells
n_pseudo_per_epoch = int(math.ceil(
    pseudo_weighted_cell_target /
    (PSEUDO_SAMPLE_WEIGHT * avg_pseudo_observed_cells)
))
n_pseudo_per_epoch = min(max(n_pseudo_per_epoch, 1), len(pseudo_train_ids))
actual_pseudo_mass_ratio = (
    n_pseudo_per_epoch * PSEUDO_SAMPLE_WEIGHT * avg_pseudo_observed_cells
    / gold_observed_cells
)

print("Gold studies per epoch:", len(gold_train_ids))
print("Pseudo studies sampled per epoch:", n_pseudo_per_epoch, "/", len(pseudo_train_ids))
print("Gold observed target cells:", int(gold_observed_cells))
print("Average observed pseudo target cells/study:", round(avg_pseudo_observed_cells, 2))
print("Expected pseudo weighted mass / gold mass:", round(actual_pseudo_mass_ratio, 3))


class GoldPrioritizedEpochSampler(Sampler):
    def __init__(self, gold_positions, pseudo_positions, n_pseudo, seed=SEED):
        self.gold_positions = np.asarray(gold_positions, dtype=np.int64)
        self.pseudo_positions = np.asarray(pseudo_positions, dtype=np.int64)
        self.n_pseudo = int(n_pseudo)
        self.seed = int(seed)
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        if self.n_pseudo <= len(self.pseudo_positions):
            pseudo_selected = rng.choice(
                self.pseudo_positions, size=self.n_pseudo, replace=False
            )
        else:
            pseudo_selected = rng.choice(
                self.pseudo_positions, size=self.n_pseudo, replace=True
            )
        selected = np.concatenate([self.gold_positions, pseudo_selected])
        rng.shuffle(selected)
        return iter(selected.tolist())

    def __len__(self):
        return len(self.gold_positions) + self.n_pseudo


train_dataset = StudyMRIDataset(train_pool_ids)
train_sampler = GoldPrioritizedEpochSampler(
    train_pool_local_gold,
    train_pool_local_pseudo,
    n_pseudo_per_epoch,
)
val_dataset = StudyMRIDataset(val_study_ids)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
    drop_last=False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

print("Dataset pool:", len(train_dataset))
print("Batches per epoch:", len(train_loader))
print("Validation studies:", len(val_dataset))

## 20. Per-target positive weights

In [ ]:
# Estimate class-imbalance correction from the full training pool, while applying
# the explicit gold/pseudo study weights. The validation loss does not use pos_weight.
train_masks_all = mask_matrix[train_pool_global_idx]
train_labels_all = y_matrix[train_pool_global_idx]
train_sample_weights_all = np.where(
    source_names[train_pool_global_idx] == "gold",
    GOLD_SAMPLE_WEIGHT,
    PSEUDO_SAMPLE_WEIGHT,
).astype(np.float32)
weighted_mask_all = train_masks_all * train_sample_weights_all[:, None]
pos_count = (train_labels_all * weighted_mask_all).sum(axis=0)
neg_count = ((1.0 - train_labels_all) * weighted_mask_all).sum(axis=0)
pos_weight_np = np.clip(
    neg_count / np.maximum(pos_count, 1e-6), 1.0, 10.0
).astype(np.float32)
pos_weight = torch.tensor(pos_weight_np, dtype=torch.float32, device=device)

display(pd.DataFrame({
    "target": TARGETS,
    "weighted_positive": pos_count,
    "weighted_negative": neg_count,
    "pos_weight": pos_weight_np,
}))

## 21. Correct masked weighted BCE

Đây là sửa lỗi quan trọng so với notebook ViT cũ.

Công thức:

\[
L = \frac{\sum_{i,j} m_{ij}\,w_i\,\operatorname{BCE}(z_{ij},y_{ij})}
{\sum_{i,j}m_{ij}}
\]

- `m_ij = 0` với uncertain / not_mentioned / missing labels.
- `w_i = 1.0` cho gold và `0.05` cho pseudo.
- **Denominator là số observed target cells không có trọng số**, vì vậy `0.05` không bị triệt tiêu như công thức cũ chia cho `sum(mask × weight)`.

Kết hợp sampler ở Cell 19 để weighted pseudo-label mass mỗi epoch ở khoảng 50% gold mass.

In [ ]:
def masked_weighted_bce_loss(logits, targets, masks, sample_w, pos_w):
    element_loss = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none",
        pos_weight=pos_w,
    )
    weighted_mask = masks * sample_w.unsqueeze(1)
    denominator = masks.sum().clamp_min(1.0)  # deliberately NOT weighted_mask.sum()
    return (element_loss * weighted_mask).sum() / denominator

# Sanity check: a pseudo-only batch must produce a loss 0.05x the unweighted loss.
_test_logits = torch.zeros((1, len(TARGETS)), device=device)
_test_targets = torch.zeros_like(_test_logits)
_test_masks = torch.ones_like(_test_logits)
_test_posw = torch.ones(len(TARGETS), device=device)
_test_full = masked_weighted_bce_loss(
    _test_logits, _test_targets, _test_masks,
    torch.ones(1, device=device), _test_posw
)
_test_pseudo = masked_weighted_bce_loss(
    _test_logits, _test_targets, _test_masks,
    torch.full((1,), PSEUDO_SAMPLE_WEIGHT, device=device), _test_posw
)
print("Gold loss:", float(_test_full), "| pseudo loss:", float(_test_pseudo))
assert abs(float(_test_pseudo / _test_full) - PSEUDO_SAMPLE_WEIGHT) < 1e-5
print("Weighted-loss sanity check: PASS")

## 22. Gold-only validation function

In [ ]:
@torch.no_grad()
def run_validation(model, loader):
    model.eval()
    weighted_loss_sum = 0.0
    observed_count = 0.0
    all_logits, all_labels, all_masks, all_uids = [], [], [], []

    for batch in loader:
        images = batch["images"].to(device, non_blocking=True)
        valid = batch["valid"].to(device, non_blocking=True)
        plane_ids = batch["plane_ids"].to(device, non_blocking=True)
        fluid = batch["fluid"].to(device, non_blocking=True)
        fat = batch["fat"].to(device, non_blocking=True)
        positions = batch["positions"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        masks = batch["label_mask"].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(images, valid, plane_ids, fluid, fat, positions)
            loss = masked_weighted_bce_loss(
                logits, labels, masks,
                torch.ones(len(batch["uid"]), device=device),
                torch.ones(len(TARGETS), device=device),
            )

        n_obs = float(masks.sum().item())
        weighted_loss_sum += float(loss.item()) * n_obs
        observed_count += n_obs
        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        all_masks.append(masks.cpu().numpy())
        all_uids.extend(batch["uid"])

    return (
        weighted_loss_sum / max(observed_count, 1.0),
        np.concatenate(all_logits),
        np.concatenate(all_labels),
        np.concatenate(all_masks),
        all_uids,
    )

print("Gold-only validation ready.")

## 23. Optimizer, scheduler và checkpoint paths

In [ ]:
BEST_CKPT = OUTPUT_ROOT / "vit_pretrained_best_checkpoint.pt"
LAST_CKPT = OUTPUT_ROOT / "vit_pretrained_last_checkpoint.pt"
HISTORY_CSV = OUTPUT_ROOT / "vit_pretrained_training_history.csv"

backbone_trainable_params = [
    p for p in model.slice_encoder.backbone.parameters() if p.requires_grad
]
new_head_params = [
    p for module in [model.slice_encoder.patch_attention, model.slice_encoder.feature_projection,
                     model.aggregator, model.classifier]
    for p in module.parameters() if p.requires_grad
]

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_trainable_params, "lr": LR_BACKBONE},
        {"params": new_head_params, "lr": LR_HEAD},
    ],
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=1, min_lr=1e-7
)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

run_config = {
    "seed": SEED,
    "architecture": "Pretrained ViT-B/16 + gated metadata + study Transformer + residual mean-attention pooling",
    "pretrained_source": PRETRAINED_SOURCE,
    "image_size": IMAGE_SIZE,
    "patch_size": PATCH_SIZE,
    "freeze_early_blocks": FREEZE_EARLY_BLOCKS,
    "slice_chunk_size": SLICE_CHUNK_SIZE,
    "embed_dim": EMBED_DIM,
    "study_depth": STUDY_DEPTH,
    "study_heads": STUDY_HEADS,
    "use_center_prior": USE_CENTER_PRIOR,
    "center_prior_strength": CENTER_PRIOR_STRENGTH,
    "center_prior_sigma": CENTER_PRIOR_SIGMA,
    "gold_sample_weight": GOLD_SAMPLE_WEIGHT,
    "pseudo_sample_weight": PSEUDO_SAMPLE_WEIGHT,
    "pseudo_to_gold_weighted_mass_ratio": PSEUDO_TO_GOLD_WEIGHTED_MASS_RATIO,
    "pseudo_samples_per_epoch": n_pseudo_per_epoch,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "epochs": EPOCHS,
    "lr_backbone": LR_BACKBONE,
    "lr_head": LR_HEAD,
    "split_source": "Notebook 04 shared_split.csv",
    "train_uid_sha": hashlib.sha256("\n".join(gold_train_ids + pseudo_train_ids).encode()).hexdigest(),
    "val_uid_sha": hashlib.sha256("\n".join(val_study_ids).encode()).hexdigest(),
}
run_signature = hashlib.sha256(
    json.dumps(run_config, sort_keys=True).encode()
).hexdigest()[:20]

print("Run signature:", run_signature)
print("Best checkpoint:", BEST_CKPT)
print("Last checkpoint:", LAST_CKPT)

## 24. Resume state (không resume nhầm cấu hình khác)

In [ ]:
start_epoch = 0
best_val_loss = float("inf")
best_epoch = -1
early_stop_counter = 0
history = []
resume_already_stopped = False

if RESUME_TRAINING and LAST_CKPT.exists():
    try:
        checkpoint = torch.load(LAST_CKPT, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(LAST_CKPT, map_location="cpu")

    if checkpoint.get("run_signature") != run_signature:
        raise RuntimeError(
            "Checkpoint tồn tại nhưng run_signature khác cấu hình hiện tại. "
            "Để tránh resume nhầm, hãy đổi OUTPUT_ROOT hoặc xóa checkpoint của run cũ."
        )

    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    if checkpoint.get("scaler_state"):
        scaler.load_state_dict(checkpoint["scaler_state"])
    start_epoch = int(checkpoint["epoch"]) + 1
    best_val_loss = float(checkpoint.get("best_val_loss", best_val_loss))
    best_epoch = int(checkpoint.get("best_epoch", best_epoch))
    early_stop_counter = int(checkpoint.get("early_stop_counter", 0))
    history = checkpoint.get("history", [])
    resume_already_stopped = early_stop_counter >= EARLY_STOPPING_PATIENCE
    print("Restored checkpoint; next epoch:", start_epoch + 1)
    print("Early-stop counter:", early_stop_counter)
    if resume_already_stopped:
        print("Previous run already reached early stopping. Training will be skipped.")
else:
    print("Starting a new pretrained ViT run.")

print("Trainable backbone params:", sum(p.numel() for p in backbone_trainable_params) / 1e6, "M")
print("New head params:", sum(p.numel() for p in new_head_params) / 1e6, "M")

## 25. One training epoch

In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, epoch):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    running_batches = 0

    for step, batch in enumerate(loader, start=1):
        images = batch["images"].to(device, non_blocking=True)
        valid = batch["valid"].to(device, non_blocking=True)
        plane_ids = batch["plane_ids"].to(device, non_blocking=True)
        fluid = batch["fluid"].to(device, non_blocking=True)
        fat = batch["fat"].to(device, non_blocking=True)
        positions = batch["positions"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        masks = batch["label_mask"].to(device, non_blocking=True)
        sample_w = batch["sample_weight"].to(device, non_blocking=True)

        # Normalize a final, incomplete accumulation group by its actual size.
        group_start = ((step - 1) // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS + 1
        group_size = min(GRAD_ACCUM_STEPS, len(loader) - group_start + 1)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(images, valid, plane_ids, fluid, fat, positions)
            loss = masked_weighted_bce_loss(
                logits, labels, masks, sample_w, pos_weight
            )
            scaled_loss = loss / group_size

        scaler.scale(scaled_loss).backward()
        should_step = (step % GRAD_ACCUM_STEPS == 0) or (step == len(loader))
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += float(loss.detach().item())
        running_batches += 1
        if step % 50 == 0 or step == len(loader):
            print(
                f"Epoch {epoch + 1}/{EPOCHS} | step {step}/{len(loader)} "
                f"| weighted_train_loss={running_loss / max(running_batches, 1):.5f}",
                flush=True,
            )

    return running_loss / max(running_batches, 1)

print("train_one_epoch() ready.")

## 26. Training loop + checkpointing + early stopping

In [ ]:
if not resume_already_stopped:
    for epoch in range(start_epoch, EPOCHS):
        train_sampler.set_epoch(epoch)
        epoch_started = time.time()

        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, epoch)
        val_loss, val_logits, val_labels, val_masks, val_uids = run_validation(model, val_loader)
        scheduler.step(val_loss)
        elapsed = time.time() - epoch_started

        history.append({
            "epoch": epoch + 1,
            "weighted_train_loss": train_loss,
            "val_bce_loss": val_loss,
            "lr_backbone": optimizer.param_groups[0]["lr"],
            "lr_head": optimizer.param_groups[1]["lr"],
            "seconds": elapsed,
        })
        pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)

        improved = val_loss < (best_val_loss - MIN_DELTA)
        if improved:
            best_val_loss = val_loss
            best_epoch = epoch
            early_stop_counter = 0
            torch.save({
                "model_state": model.state_dict(),
                "targets": TARGETS,
                "run_config": run_config,
                "run_signature": run_signature,
                "best_epoch": best_epoch + 1,
                "best_val_loss": best_val_loss,
            }, BEST_CKPT)
        else:
            early_stop_counter += 1

        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "best_val_loss": best_val_loss,
            "best_epoch": best_epoch,
            "early_stop_counter": early_stop_counter,
            "history": history,
            "run_config": run_config,
            "run_signature": run_signature,
        }, LAST_CKPT)

        print(
            f"Epoch {epoch + 1}: weighted_train_loss={train_loss:.5f} | "
            f"val_bce_loss={val_loss:.5f} | best={best_val_loss:.5f} "
            f"(epoch {best_epoch + 1}) | {elapsed / 60:.1f} min",
            flush=True,
        )

        if early_stop_counter >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break
else:
    print("Skip training: this checkpoint already completed early stopping.")

if not BEST_CKPT.exists():
    raise FileNotFoundError("Best checkpoint was not created. Inspect training logs.")
print("Training finished. Best checkpoint:", BEST_CKPT)

## 27. Review training history

In [ ]:
if HISTORY_CSV.exists():
    history_df = pd.read_csv(HISTORY_CSV)
    display(history_df)
    if len(history_df):
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(history_df["epoch"], history_df["weighted_train_loss"], marker="o", label="Weighted train loss")
        ax.plot(history_df["epoch"], history_df["val_bce_loss"], marker="o", label="Gold validation BCE")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title("Training history")
        ax.legend()
        ax.grid(alpha=0.2)
        plt.show()
else:
    print("No history CSV exists yet.")

## 28. Load best checkpoint and run gold validation

In [ ]:
try:
    best_checkpoint = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
except TypeError:
    best_checkpoint = torch.load(BEST_CKPT, map_location="cpu")

if best_checkpoint.get("targets") != TARGETS:
    raise ValueError("Checkpoint target order does not match TARGETS")
model.load_state_dict(best_checkpoint["model_state"])
model.to(device).eval()

val_loss, val_logits, val_labels_np, val_masks_np, val_uids = run_validation(model, val_loader)
val_probs = 1.0 / (1.0 + np.exp(-val_logits))
print("Best epoch:", best_checkpoint.get("best_epoch"))
print("Gold validation BCE:", round(float(val_loss), 5))

## 29. Per-target metrics và validation predictions

In [ ]:
metric_rows = []
wide_rows = []
long_rows = []

for j, target in enumerate(TARGETS):
    valid = val_masks_np[:, j] > 0
    y_true = val_labels_np[valid, j].astype(int)
    y_prob = val_probs[valid, j]
    y_pred = (y_prob >= 0.5).astype(int)
    has_two_classes = len(np.unique(y_true)) == 2
    auc = roc_auc_score(y_true, y_prob) if has_two_classes else np.nan

    metric_rows.append({
        "target": target,
        "n_validation": len(y_true),
        "n_positive": int(y_true.sum()) if len(y_true) else 0,
        "n_negative": int((1 - y_true).sum()) if len(y_true) else 0,
        "roc_auc": auc,
        "f1_at_0.5": f1_score(y_true, y_pred, zero_division=0) if len(y_true) else np.nan,
        "accuracy_at_0.5": accuracy_score(y_true, y_pred) if len(y_true) else np.nan,
        "auc_defined": bool(has_two_classes),
    })

    for i, uid in enumerate(val_uids):
        wide_rows.append({
            ID_COL: uid,
            f"{target}__y_true": int(val_labels_np[i, j]),
            f"{target}__probability": float(val_probs[i, j]),
        })
        long_rows.append({
            ID_COL: uid,
            "target": target,
            "y_true": int(val_labels_np[i, j]),
            "probability": float(val_probs[i, j]),
            "prediction_at_0.5": int(val_probs[i, j] >= 0.5),
        })

metrics_df = pd.DataFrame(metric_rows)
wide_predictions_df = pd.DataFrame(wide_rows).groupby(ID_COL, as_index=False).first()
long_predictions_df = pd.DataFrame(long_rows)

metrics_path = OUTPUT_ROOT / "vit_pretrained_validation_metrics.csv"
wide_predictions_path = OUTPUT_ROOT / "vit_pretrained_validation_predictions_wide.csv"
long_predictions_path = OUTPUT_ROOT / "vit_pretrained_validation_predictions_long.csv"
metrics_df.to_csv(metrics_path, index=False)
wide_predictions_df.to_csv(wide_predictions_path, index=False)
long_predictions_df.to_csv(long_predictions_path, index=False)

valid_auc = metrics_df["roc_auc"].dropna()
print("Mean defined per-target ROC-AUC:", round(float(valid_auc.mean()), 4) if len(valid_auc) else "undefined")
print("Defined targets:", len(valid_auc), "/", len(TARGETS))
display(metrics_df)
print("CAUTION: validation has only 12 studies; use this table as a diagnostic, not a stable final estimate.")

## 30. Experiment manifest

In [ ]:
run_manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "architecture": run_config["architecture"],
    "pretrained": True,
    "pretrained_source": PRETRAINED_SOURCE,
    "source_cache": str(CACHE_DIR),
    "source_cnn_artifact_zip": str(ARTIFACT_ZIP),
    "n_total_studies": len(uid_list),
    "n_gold_total": n_gold,
    "n_pseudo_total": n_pseudo,
    "n_gold_train": len(gold_train_ids),
    "n_pseudo_pool_train": len(pseudo_train_ids),
    "n_pseudo_sampled_per_epoch": n_pseudo_per_epoch,
    "n_gold_validation": len(val_study_ids),
    "image_size": IMAGE_SIZE,
    "patch_size": PATCH_SIZE,
    "max_slices": MAX_SLICES,
    "freeze_early_blocks": FREEZE_EARLY_BLOCKS,
    "use_center_prior": USE_CENTER_PRIOR,
    "center_prior_strength": CENTER_PRIOR_STRENGTH,
    "center_prior_sigma": CENTER_PRIOR_SIGMA,
    "gold_sample_weight": GOLD_SAMPLE_WEIGHT,
    "pseudo_sample_weight": PSEUDO_SAMPLE_WEIGHT,
    "pseudo_to_gold_weighted_mass_ratio": PSEUDO_TO_GOLD_WEIGHTED_MASS_RATIO,
    "actual_expected_pseudo_mass_ratio": actual_pseudo_mass_ratio,
    "best_epoch": int(best_checkpoint.get("best_epoch", -1)),
    "best_validation_bce": float(best_checkpoint.get("best_val_loss", np.nan)),
    "mean_defined_validation_auc": float(valid_auc.mean()) if len(valid_auc) else None,
    "defined_auc_targets": int(len(valid_auc)),
    "split_source": "Notebook 04 shared_split.csv",
    "label_policy": "masked BCE; uncertain/not_mentioned ignored",
    "weighted_loss_denominator": "unweighted observed target-cell count",
    "validation_warning": "Only 12 gold validation studies; high uncertainty",
    "run_config": run_config,
}

manifest_path = OUTPUT_ROOT / "vit_pretrained_manifest.json"
manifest_path.write_text(
    json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Manifest:", manifest_path)

## 31. Export artifacts để dùng cho submission / model comparison

In [ ]:
artifact_paths = [
    BEST_CKPT,
    LAST_CKPT,
    HISTORY_CSV,
    metrics_path,
    wide_predictions_path,
    manifest_path,
    ARTIFACT_DIR / "shared_split.csv",
    ARTIFACT_DIR / "study_labels_training.csv",
    ARTIFACT_DIR / "preprocess_status.csv",
]

vit_zip = OUTPUT_ROOT / "vit_pretrained_centerprior_model_artifacts.zip"
with zipfile.ZipFile(vit_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in artifact_paths:
        if path.exists():
            zf.write(path, arcname=path.name)

print("Pretrained ViT artifact ZIP:", vit_zip)
print("Cache is reused from Notebook 04 and is not duplicated in this ZIP.")
print("ZIP size GiB:", round(vit_zip.stat().st_size / 1024**3, 3))
display(FileLink(vit_zip))

## 32. Checklist trước khi lưu Kaggle Version

- [ ] Pretrained source được in rõ và là ImageNet-1K ViT-B/16; không có random-init fallback.
- [ ] Cache shape là `(4407, 36, 224, 224)`; model không resize ảnh xuống 160.
- [ ] `shared_split.csv` giống Notebook 04; validation vẫn 12 gold UIDs.
- [ ] Weighted-loss sanity check báo pseudo loss bằng `0.05 ×` gold loss trên cùng logits/labels.
- [ ] Pseudo subset size và `actual_expected_pseudo_mass_ratio` được in ra; dự kiến gần 0.50.
- [ ] Center prior là soft additive bias, không hard-mask vùng ngoại vi.
- [ ] Lưu output của version có chứa `vit_pretrained_centerprior_model_artifacts.zip`.

**Lưu ý nghiên cứu:** center prior là giả thuyết cần kiểm chứng. Một số bất thường và cấu trúc giải phẫu có thể nằm lệch về hai bên hoặc gần rìa FOV; prior được giữ nhẹ để tránh ép model bỏ qua chúng. Sau run đầu, nên làm ablation `USE_CENTER_PRIOR=False` với cùng split, seed và protocol để biết center prior thực sự có ích hay không.